# Hybrid RAG

Combining the [langchain-rag](https://python.langchain.com/docs/tutorials/rag/) with the 
[hybrid rag](https://pub.towardsai.net/hybrid-rag-made-easy-step-by-step-with-langchain-faiss-azureopenai-llmgraphtransformer-and-ef93cd50948d)

In [2]:
import os
from langchain_experimental.graph_transformers import LLMGraphTransformer
import networkx as nx
from langchain.chains import GraphQAChain
from langchain_core.documents import Document
from langchain_community.graphs.networkx_graph import NetworkxEntityGraph
from langchain.chains import RetrievalQA
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore

from langchain.vectorstores import FAISS
from langchain.document_loaders import TextLoader
from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings
import json
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
import pandas as pd
from langchain_core.documents import Document
from langchain_ollama import ChatOllama
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain.prompts import ChatPromptTemplate
from langchain_core.prompts import PromptTemplate



## Set up the basic RAG

### create the vector store

In [8]:

llm_model = "llama3.2:latest"
llm = ChatOllama(
   model=llm_model,
   temperature=0,
   # other params...
)


def build_vector_store(documents,filename,embeddings):
    
    vector_store = Chroma(
    collection_name="example_collection",
        embedding_function=embeddings,
     persist_directory="./"+filename,  # Where to save data locally, remove if not necessary
     )
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    all_splits = text_splitter.split_documents(training_documents)
    _ = vector_store.add_documents(documents=all_splits)
    return vector_store

print("starting to load")
with open("./data/training_modeling_papers.json", "r") as f:
    data = json.load(f)

print("read json..."+str(len(data)))
training_documents = []

for row in data:
    training_documents.append(Document(page_content=row["abstract"]))

print("documents created..."+str(len(training_documents)))
f"Papers loaded: {len(training_documents)}"

embeddings = OllamaEmbeddings(model=llm_model)

vector_store = build_vector_store(training_documents,"chroma_langchain.db",embeddings)


starting to load
read json...46
documents created...46


### build the RAG retrieval

In [9]:

def create_generic_rag(vector_store):
    template = """Use the following pieces of context to summarize the question provided at the end.

    {context}

    Question: {question}

    Helpful Answer:"""

    custom_rag_prompt = PromptTemplate.from_template(template)

    # set up state
    # Define state for application
    class State(TypedDict):
        question: str
        context: List[Document]
        answer: str


    # Define application steps
    def retrieve(state: State):
        retrieved_docs = vector_store.similarity_search(state["question"])
        return {"context": retrieved_docs}


    def generate(state: State):
        docs_content = "\n\n".join(doc.page_content for doc in state["context"])
        messages = custom_rag_prompt.invoke({"question": state["question"], "context": docs_content})
        response = llm.invoke(messages)
        return {"answer": response.content}

    # Compile application and test
    graph_builder = StateGraph(State).add_sequence([retrieve, generate])
    graph_builder.add_edge(START, "retrieve")
    generic_rag = graph_builder.compile()
    return generic_rag

In [10]:
generic_rag = create_generic_rag(vector_store)
response = generic_rag.invoke({"question": "What countries did COVID occur in?"})

## build the graphrag

In [11]:
def build_graph_rag(llm,training_documents):

    llm_transformer = LLMGraphTransformer(llm=llm)
    graph_documents = llm_transformer.convert_to_graph_documents(training_documents)

    graph = NetworkxEntityGraph()

    for node in graph_documents[0].nodes:
        graph.add_node(node.id)

    for edge in graph_documents[0].relationships:
        graph._graph.add_edge(
            edge.source.id,
            edge.target.id,
            relation=edge.type
        )

        graph._graph.add_edge(
            edge.target.id,
            edge.source.id,
            relation=edge.type+" by",
        )
    graph_rag = GraphQAChain.from_llm(
            llm=llm,
            graph=graph,
            verbose=True
        )
    return graph_rag

In [12]:
graph_rag = build_graph_rag(llm,training_documents)

In [13]:
def hybrid_rag(generic_rag,graph_rag):
    def rag_processor(query):
        # Doing generic RAG
        rag_result = generic_rag.invoke({"question": query})
    
        # Printing the generic RAG Results
      #  print("----------------------------------------------")
       # print("Generic RAG Result - ",rag_result)
        #print("----------------------------------------------")
    
        # Doing GraphRAG
        graph_rag_result = graph_rag.invoke({"query":query})
    
        # Printing the GraphRAG Results
        #print("----------------------------------------------")
        #print("Graph RAG Result - ",graph_rag_result)
        #print("----------------------------------------------")

        #prompt = """You are a helpful assistant.
        #Generate an ultimate response of the question provided by combining the 
        #two contexts provided :
        prompt = """You are a helpful assistant.
        Generate a summary of the passage provided, using the two contexts provided:

        Context 1: {}
        Context 2: {} 

        Question: {}
    
        """.format(rag_result,graph_rag_result,query)

        return llm.invoke(prompt),rag_result,graph_rag_result
    return rag_processor

In [14]:
hr = hybrid_rag(generic_rag,graph_rag)

In [16]:
def print_graph_results(graph_documents: list[Document]) -> None:
    for doc in graph_documents:
        if len(doc.nodes) > 0:
            print(f"Paper ID: {doc.source.id}")
            print(f"Paper Abstract: {doc.source.page_content}")

            for node in doc.nodes:
                print(node)
                print(f"Node: {node.id}, Type: {node.type}")

            for rel in doc.relationships:
                print(f"Relationship: {rel.type}")
                print(f"   Source: {rel.source.id}, Type: {rel.source.type}")
                print(f"   Target: {rel.target.id}, Type: {rel.target.type}")

            print()


def pgr(graph_documents: list[Document]) -> None:
    for doc in graph_documents:
        if len(doc.nodes) > 0 and len(doc.relationships) > 0:
            for rel in doc.relationships:
                print(f"   Source: {rel.source.id} ({rel.source.type}) -> {rel.type} -> {rel.target.id} ({rel.target.type})")
        print()

In [17]:
import pandas as pd

df_modeling_papers = pd.read_json("./data/modeling_papers_0.json", orient="records", lines=True)

documents = []

for row in df_modeling_papers.itertuples():
    documents.append(Document(id=row.id, page_content=row.abstract))

f"Papers loaded: {len(documents)}"

'Papers loaded: 5737'

In [18]:
documents[0]

Document(id='37227da2b75373b500a6a9f24649dcec', metadata={}, page_content='Many applications in science and engineering involve data defined at specific geospatial locations, which are often modeled as random fields. The modeling of a proper correlation function is essential for the probabilistic calibration of the random fields, but traditional methods were developed with the assumption to have observations with evenly spaced data. Available methods dealing with irregularly spaced data generally require either interpolation or computationally expensive solutions. Instead, we propose a simple approach based on least square regression to estimate the autocorrelation function. We first tested our methodology on an artificially produced dataset to assess the performance of our method. The accuracy of the method and its robustness to the level of noise in the data indicate that it is suitable for use in realistic problems. In addition, the methodology was used on a major application, the m

In [19]:
res,rag_res,graph_res = hr(documents[0].page_content)



> Entering new GraphQAChain chain...
Entities Extracted:
 NONE
Full Context:


> Finished chain.


In [20]:
res

AIMessage(content='Based on Context 1 and Context 2, I will provide a summary of the passage:\n\nThe passage discusses the importance of modeling correlation functions for probabilistic calibration of random fields in science and engineering. Traditional methods assume evenly spaced data but are not suitable for irregularly spaced data. A new approach based on least square regression is proposed to estimate autocorrelation functions. The method was tested on an artificially produced dataset and shown to be accurate and robust. It was also applied to a major application, the modeling of animal species connected with zoonotic diseases, such as bats. The study found that the bare carrying capacity of bats was denser in central Africa due to climatic and environmental conditions.\n\nThe passage highlights the potential of this approach for coping with large gaps in data and complex applications involving large geographical areas and high resolution. It suggests that this method can be used

In [21]:
res.content

'Based on Context 1 and Context 2, I will provide a summary of the passage:\n\nThe passage discusses the importance of modeling correlation functions for probabilistic calibration of random fields in science and engineering. Traditional methods assume evenly spaced data but are not suitable for irregularly spaced data. A new approach based on least square regression is proposed to estimate autocorrelation functions. The method was tested on an artificially produced dataset and shown to be accurate and robust. It was also applied to a major application, the modeling of animal species connected with zoonotic diseases, such as bats. The study found that the bare carrying capacity of bats was denser in central Africa due to climatic and environmental conditions.\n\nThe passage highlights the potential of this approach for coping with large gaps in data and complex applications involving large geographical areas and high resolution. It suggests that this method can be used to predict and pr

In [22]:
example_out = [Document(page_content=res.content)]
graph_documents = transformer.convert_to_graph_documents(example_out)
print_graph_results(graph_documents)

Paper ID: None
Paper Abstract: Based on Context 1 and Context 2, I will provide a summary of the passage:

The passage discusses the importance of modeling correlation functions for probabilistic calibration of random fields in science and engineering. Traditional methods assume evenly spaced data but are not suitable for irregularly spaced data. A new approach based on least square regression is proposed to estimate autocorrelation functions. The method was tested on an artificially produced dataset and shown to be accurate and robust. It was also applied to a major application, the modeling of animal species connected with zoonotic diseases, such as bats. The study found that the bare carrying capacity of bats was denser in central Africa due to climatic and environmental conditions.

The passage highlights the potential of this approach for coping with large gaps in data and complex applications involving large geographical areas and high resolution. It suggests that this method can

In [23]:
documents[2]

Document(id='02f814e361d140f018b9abff70f2ade6', metadata={}, page_content='In recent studies of influenza vaccine effectiveness (VE), lower effectiveness with increasing time since vaccination was observed, raising the question of optimal vaccination timing. We sought to evaluate the estimated number of influenza-associated hospitalizations among older adults due to potential changes in vaccination timing.Using empirical data and a health state transition model, we estimated change in influenza-associated hospitalizations predicted to occur among the US population aged ≥65 years if vaccination were delayed until October 1. We assumed the vaccination timing, coverage, and effectiveness observed in 2012-2013 as a prototypical influenza season, approximately 7% monthly waning of VE, and that between 0% and 50% of individuals who usually get vaccinated earlier than October failed to get vaccinated. We also assessed change in influenza-associated hospitalizations if vaccination uptake shift

In [24]:
res,rag_res,graph_res = hr(documents[2].page_content)



> Entering new GraphQAChain chain...
Entities Extracted:
US, October, August, September, 2012-2013
Full Context:


> Finished chain.


In [25]:
doc2 = [documents[2]]
graph_documents = transformer.convert_to_graph_documents(doc2)
print_graph_results(graph_documents)

Paper ID: 02f814e361d140f018b9abff70f2ade6
Paper Abstract: In recent studies of influenza vaccine effectiveness (VE), lower effectiveness with increasing time since vaccination was observed, raising the question of optimal vaccination timing. We sought to evaluate the estimated number of influenza-associated hospitalizations among older adults due to potential changes in vaccination timing.Using empirical data and a health state transition model, we estimated change in influenza-associated hospitalizations predicted to occur among the US population aged ≥65 years if vaccination were delayed until October 1. We assumed the vaccination timing, coverage, and effectiveness observed in 2012-2013 as a prototypical influenza season, approximately 7% monthly waning of VE, and that between 0% and 50% of individuals who usually get vaccinated earlier than October failed to get vaccinated. We also assessed change in influenza-associated hospitalizations if vaccination uptake shifted substantially

In [26]:
pgr(graph_documents)

   Source: Influenza_Vaccine_Effectiveness (Concept) -> RELATIONSHIP -> Ve_Waning (Relationship)
   Source: Vaccination_Timing (Concept) -> RELATIONSHIP -> Influenza_Season_Timing (Relationship)
   Source: Health_State_Transition_Model (Concept) -> RELATIONSHIP -> Us_Population (Entity)
   Source: Influenza-Associated_Hospitalizations (Concept) -> RELATIONSHIP -> Older_Adults (Entity)
   Source: Prototypical_Influenza_Season (Concept) -> RELATIONSHIP -> Ve_Waning (Relationship)
   Source: August_Vaccination_Shift (Concept) -> RELATIONSHIP -> Influenza-Associated_Hospitalizations (Concept)
   Source: September_Vaccination_Shift (Concept) -> RELATIONSHIP -> Influenza-Associated_Hospitalizations (Concept)



In [27]:
example_out = [Document(page_content=res.content)]
graph_documents = transformer.convert_to_graph_documents(example_out)
pgr(graph_documents)

   Source: Influenza (Disease) -> CAUSES -> Hospitalization (Condition)
   Source: Vaccination (Concept) -> PREVENTS -> Older Adults (Group)
   Source: August (Month) -> OCCURS_IN -> Older Adults (Group)
   Source: September (Month) -> OCCURS_IN -> Older Adults (Group)
   Source: October (Month) -> STARTS -> Vaccination (Concept)

